# LeetCode 2744 — Find Maximum Number of String Pairs bằng Python

Link đề: https://leetcode.com/problems/find-maximum-number-of-string-pairs/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def maximumNumberOfStringPairs(self, words: List[str]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **cặp** `(i, j)` với `i < j` sao cho `words[i]` là **chuỗi đảo ngược** của `words[j]` (ví dụ `"cd"` và `"dc"`).
2. Nắm các công cụ nền của Python: **chỉ số chuỗi**, **cắt lát đảo ngược `s[::-1]`**, **`set`** và toán tử `in`, hai vòng `for` lồng nhau với `range(i + 1, n)`.
3. Đi từ **Brute Force O(n²)** (thử mọi cặp) → **`set` đã gặp (O(n))** → **đếm bằng điều kiện `w < reverse(w)`** → **bảng 26×26** → **`Counter`** (mở rộng cho trường hợp có chuỗi trùng).
4. Hiểu vì sao đề cho các chuỗi **đôi một khác nhau**: mỗi chuỗi có tối đa **một** bạn đảo ngược, nên "ghép tối đa" chỉ là "đếm tất cả cặp".
5. Tránh các lỗi: đếm một cặp hai lần, ghép chuỗi đối xứng như `"zz"` với chính nó, thêm vào `set` trước khi kiểm tra, và dùng `reversed(w)` (trả về iterator, không phải chuỗi).

## 2. Đề bài & ví dụ

Cho mảng `words` gồm các chuỗi **đôi một khác nhau**, **mỗi chuỗi dài đúng 2 ký tự**.

Hai chuỗi `words[i]` và `words[j]` tạo thành một **cặp** nếu:

- `0 <= i < j < words.length`, và
- `words[i]` bằng **chuỗi đảo ngược** của `words[j]`.

Trả về **số cặp tối đa** có thể tạo từ mảng `words`. Lưu ý mỗi chuỗi chỉ thuộc **tối đa một cặp**.

| Ví dụ | words | Output | Giải thích |
|---|---|---|---|
| 1 | `["cd","ac","dc","ca","zz"]` | `2` | Cặp `(0, 2)`: `"cd"` và `"dc"`; cặp `(1, 3)`: `"ac"` và `"ca"`. `"zz"` không có bạn. |
| 2 | `["ab","ba","cc"]` | `1` | Cặp `(0, 1)`: `"ab"` và `"ba"`. |
| 3 | `["aa","ab"]` | `0` | `"aa"` đảo ngược vẫn là `"aa"` nhưng không có chuỗi `"aa"` thứ hai (các chuỗi khác nhau). |

Constraints:

- `1 <= words.length <= 50`
- `words[i].length == 2`
- `words` chỉ chứa các chuỗi **đôi một khác nhau**.
- `words[i]` chỉ gồm chữ cái tiếng Anh viết thường.

In [1]:
words = ["cd", "ac", "dc", "ca", "zz"]
for i in range(len(words)):
    for j in range(i + 1, len(words)):
        if words[i] == words[j][::-1]:
            print("cặp", (i, j), words[i], words[j])

cặp (0, 2) cd dc
cặp (1, 3) ac ca


## 3. Kiến thức nền

### 3.1. Chỉ số của chuỗi (indexing)

Chuỗi trong Python là dãy ký tự, đánh số từ `0`. `s[0]` là ký tự đầu, `s[-1]` là ký tự cuối. Với chuỗi dài đúng 2: `s[0]` và `s[1]` (hoặc `s[-1]`) là tất cả những gì có.

In [2]:
s = "cd"
print(s[0], s[1], s[-1])   # c d d
print(len(s))              # 2

c d d
2


### 3.2. Cắt lát đảo ngược `s[::-1]`

Cú pháp cắt lát: `s[bắt_đầu:kết_thúc:bước]`. Bước `-1` nghĩa là đi **lùi** từng ký tự, nên `s[::-1]` là chuỗi **đảo ngược**.

In [3]:
print("cd"[::-1])        # dc
print("abc"[::-1])       # cba
print("zz"[::-1])        # zz   (chuỗi đối xứng đảo ngược vẫn như cũ)
print("cd"[1] + "cd"[0]) # dc  (cách thủ công cho độ dài 2)

dc
cba
zz
dc


### 3.3. `set` (tập hợp) và toán tử `in`

`set` lưu các phần tử **không trùng** và cho kiểm tra `x in s` nhanh (trung bình `O(1)`). Thêm phần tử bằng `.add(x)`.

In [4]:
seen = set()
seen.add("ab")
seen.add("cd")
seen.add("ab")        # thêm lần hai không có tác dụng
print(seen)            # {'ab', 'cd'} (thứ tự không cố định)
print("ab" in seen)    # True
print("ba" in seen)    # False
print(len(seen))       # 2

{'cd', 'ab'}
True
False
2


### 3.4. Hai vòng `for` lồng nhau với `range(i + 1, n)`

Để xét **mọi cặp không thứ tự** `(i, j)` với `i < j`, cho `j` chạy từ `i + 1` tới cuối. Cách này không bao giờ lặp một cặp hai lần và không ghép phần tử với chính nó.

In [5]:
n = 4
pairs = []
for i in range(n):
    for j in range(i + 1, n):
        pairs.append((i, j))
print(pairs)        # [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
print(len(pairs), n * (n - 1) // 2)   # 6 6

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
6 6


### 3.5. So sánh chuỗi và `sum` trên giá trị `True/False`

- Chuỗi so sánh `<`, `>` theo **thứ tự từ điển**: `"ab" < "ba"` là `True`.
- `True` đếm như `1`, `False` như `0`, nên `sum(điều_kiện for ...)` đếm số phần tử thoả điều kiện.

In [6]:
print("ab" < "ba", "cd" < "dc", "zz" < "zz")   # True True False
print(sum([True, False, True]))                   # 2
print(sum(w < w[::-1] for w in ["ab", "ba", "zz"]))  # 1  (chỉ "ab")

True True False
2
1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với `words = ["cd","ac","dc","ca","zz"]`, ta "ghép đôi" như ghép găng tay trái với găng tay phải:

- `"cd"` cần bạn là `"dc"` → có ở vị trí 2 ✓
- `"ac"` cần bạn là `"ca"` → có ở vị trí 3 ✓
- `"dc"` cần bạn là `"cd"` → đã được `"cd"` ghép rồi.
- `"zz"` cần bạn là `"zz"` → không có chuỗi `"zz"` thứ hai (các chuỗi đều khác nhau) ✗.

**Nhận xét then chốt:** chuỗi `w` chỉ có thể ghép với **đúng một** chuỗi khả dĩ là `w[::-1]`. Vì mọi chuỗi trong `words` đôi một khác nhau, chuỗi `w[::-1]` xuất hiện **tối đa một lần**. Do đó không có chuyện chọn ghép "nên ghép cái nào"; đáp án chính là **số cặp (i, j) thoả điều kiện**, không cần bài toán ghép cặp tối ưu.

In [7]:
words = ["cd", "ac", "dc", "ca", "zz"]
for w in words:
    r = w[::-1]
    print(f"{w!r} cần bạn {r!r}: {'có' if r in words and r != w else 'không có'}")

'cd' cần bạn 'dc': có
'ac' cần bạn 'ca': có
'dc' cần bạn 'cd': có
'ca' cần bạn 'ac': có
'zz' cần bạn 'zz': không có


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp (i, j)

**Ý tưởng:** làm đúng như đề phát biểu:

1. Duyệt `i` từ `0` tới `n − 1`.
2. Với mỗi `i`, duyệt `j` từ `i + 1` tới `n − 1`.
3. Nếu `words[i] == words[j][::-1]` thì tăng đếm.

Vì mỗi chuỗi có tối đa một bạn, các cặp tìm được không chung phần tử, nên số cặp đếm được chính là số cặp tối đa.

In [8]:
from typing import List

class SolutionBrute:
    def maximumNumberOfStringPairs(self, words: List[str]) -> int:
        count = 0
        n = len(words)
        for i in range(n):
            for j in range(i + 1, n):
                if words[i] == words[j][::-1]:
                    count += 1
        return count

In [9]:
sol = SolutionBrute()
print(sol.maximumNumberOfStringPairs(["cd", "ac", "dc", "ca", "zz"]))   # 2
print(sol.maximumNumberOfStringPairs(["ab", "ba", "cc"]))               # 1
print(sol.maximumNumberOfStringPairs(["aa", "ab"]))                     # 0

2
1
0


**Độ phức tạp:** hai vòng lồng ⇒ `n(n − 1)/2` lần so sánh, mỗi lần so 2 ký tự (hằng số) ⇒ Time `O(n²)`, Space `O(1)` (không tính chuỗi đảo ngược tạm thời độ dài 2).

**Nút thắt:** với `n <= 50` cách này nhanh thừa. Nhưng ý "mỗi chuỗi cần xem lại cả mảng để tìm bạn" là lãng phí: câu hỏi "bạn của `w` có trong mảng không?" là bài toán **tra cứu**, mà `set` làm trong `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — tra cứu thay vì tìm kiếm.** Duyệt từ trái sang phải, giữ `seen` là tập các chuỗi đã gặp. Khi tới chuỗi `w`, nếu `w[::-1]` **đã nằm trong** `seen` thì tồn tại chuỗi đứng trước là bạn của `w` ⇒ có đúng một cặp `(i, j)` với `j` là vị trí hiện tại. Sau đó thêm `w` vào `seen`.

**Vì sao mỗi cặp đếm đúng một lần?** Cặp `(i, j)` với `i < j` chỉ được đếm ở thời điểm duyệt `j` (lúc `words[i]` đã nằm trong `seen`). Khi duyệt `i` thì `words[j]` chưa có trong `seen`.

**Đột phá 2 — "kiểm tra trước, thêm sau".** Nếu thêm `w` vào `seen` trước rồi mới kiểm tra, chuỗi đối xứng như `"zz"` sẽ tự thấy chính nó và bị đếm sai.

**Đột phá 3 — bỏ thứ tự duyệt.** Cặp `{w, rev(w)}` với `w != rev(w)` có hai thành viên; ta chỉ cần đếm thành viên nhỏ hơn theo thứ tự từ điển: `w < rev(w)`. Chuỗi đối xứng (`w == rev(w)`) bị loại tự nhiên.

In [10]:
seen = set()
count = 0
for w in ["cd", "ac", "dc", "ca", "zz"]:
    found = w[::-1] in seen
    print(f"w={w!r}, bạn={w[::-1]!r}, đã gặp bạn? {found}, seen={sorted(seen)}")
    if found:
        count += 1
    seen.add(w)
print("count =", count)

w='cd', bạn='dc', đã gặp bạn? False, seen=[]
w='ac', bạn='ca', đã gặp bạn? False, seen=['cd']
w='dc', bạn='cd', đã gặp bạn? True, seen=['ac', 'cd']
w='ca', bạn='ac', đã gặp bạn? True, seen=['ac', 'cd', 'dc']
w='zz', bạn='zz', đã gặp bạn? False, seen=['ac', 'ca', 'cd', 'dc']
count = 2


## 7. Các cách tiếp theo

### Cách 2 — `set` các chuỗi đã gặp (kiểm tra trước, thêm sau)

**Từng đoạn:**

- `seen = set()` và `count = 0`.
- Với mỗi `w`: nếu `w[::-1] in seen` thì `count += 1`.
- Cuối vòng lặp: `seen.add(w)`.

In [11]:
class SolutionSeen:
    def maximumNumberOfStringPairs(self, words: List[str]) -> int:
        seen = set()
        count = 0
        for w in words:
            if w[::-1] in seen:
                count += 1
            seen.add(w)
        return count

In [12]:
sol = SolutionSeen()
print(sol.maximumNumberOfStringPairs(["cd", "ac", "dc", "ca", "zz"]))   # 2
print(sol.maximumNumberOfStringPairs(["ab", "ba", "cc"]))               # 1
print(sol.maximumNumberOfStringPairs(["aa", "ab"]))                     # 0

2
1
0


**Code nộp LeetCode (Cách 2):** chính là `SolutionSeen`, đổi tên `Solution`.

**Độ phức tạp:** một lượt duyệt, mỗi bước `O(1)` trung bình (chuỗi dài 2) ⇒ Time `O(n)`, Space `O(n)` cho `seen`.

### Cách 3 — Chỉ đếm khi `w < đảo(w)` và đảo(w) có trong tập

**Từng đoạn:** tạo `S = set(words)` một lần. Một cặp `{w, rev(w)}` hợp lệ nếu cả hai thuộc `S` và `w != rev(w)`. Để mỗi cặp chỉ đếm một lần, chỉ xét thành viên **nhỏ hơn** về thứ tự từ điển: `w < w[::-1]`.

In [13]:
class SolutionLess:
    def maximumNumberOfStringPairs(self, words: List[str]) -> int:
        S = set(words)
        return sum(1 for w in S if w < w[::-1] and w[::-1] in S)

In [14]:
sol = SolutionLess()
print(sol.maximumNumberOfStringPairs(["cd", "ac", "dc", "ca", "zz"]))   # 2
print(sol.maximumNumberOfStringPairs(["ab", "ba", "cc"]))               # 1
print(sol.maximumNumberOfStringPairs(["aa", "ab"]))                     # 0

2
1
0


**Độ phức tạp:** Time `O(n)` (tạo set + một lượt), Space `O(n)`.

**Lưu ý:** cách này dựa vào "các chuỗi đôi một khác nhau". Nếu mảng có trùng, `set` sẽ làm mất thông tin về số lần xuất hiện (xem Cách 5).

### Cách 4 — Bảng 26 × 26 (mảng hai chiều thay cho `set`)

Mỗi chuỗi `ab` (hai chữ cái thường) tương ứng ô `(a, b)` với `a, b` thuộc `0..25`. Ta giữ `grid[a][b] = True` nếu đã gặp `ab`. Chuỗi `w = "xy"` có bạn nếu `grid[y][x]` đã `True`.

`ord(c) - ord('a')` đổi chữ cái thành số từ 0 tới 25.

In [15]:
print(ord("a"), ord("b"), ord("z"))            # 97 98 122
print(ord("c") - ord("a"), ord("z") - ord("a"))  # 2 25

97 98 122
2 25


In [16]:
class SolutionGrid:
    def maximumNumberOfStringPairs(self, words: List[str]) -> int:
        grid = [[False] * 26 for _ in range(26)]
        count = 0
        for w in words:
            x = ord(w[0]) - ord("a")
            y = ord(w[1]) - ord("a")
            if grid[y][x]:
                count += 1
            grid[x][y] = True
        return count

In [17]:
sol = SolutionGrid()
print(sol.maximumNumberOfStringPairs(["cd", "ac", "dc", "ca", "zz"]))   # 2
print(sol.maximumNumberOfStringPairs(["ab", "ba", "cc"]))               # 1
print(sol.maximumNumberOfStringPairs(["aa", "ab"]))                     # 0

2
1
0


**Độ phức tạp:** Time `O(n)`, Space `O(26 × 26) = O(1)` (hằng số). Ưu điểm: không băm chuỗi; nhược điểm: phụ thuộc bảng chữ cái cố định.

### Cách 5 — `Counter` (mở rộng cho mảng có chuỗi trùng)

Đề gốc đảm bảo các chuỗi khác nhau. Nhưng nếu **cho phép trùng**, mỗi chuỗi vẫn thuộc tối đa một cặp, và bài toán trở thành ghép tối đa:

- Với `w < rev(w)`: số cặp là `min(cnt[w], cnt[rev(w)])`.
- Với `w == rev(w)` (đối xứng): số cặp là `cnt[w] // 2`.

Với dữ liệu hợp lệ của đề (mọi `cnt = 1`), công thức này trả về cùng đáp án với các cách trên.

In [18]:
from collections import Counter

class SolutionCounter:
    def maximumNumberOfStringPairs(self, words: List[str]) -> int:
        cnt = Counter(words)
        total = 0
        for w, c in cnt.items():
            r = w[::-1]
            if w < r:
                total += min(c, cnt.get(r, 0))
            elif w == r:
                total += c // 2
        return total

In [19]:
sol = SolutionCounter()
print(sol.maximumNumberOfStringPairs(["cd", "ac", "dc", "ca", "zz"]))   # 2
print(sol.maximumNumberOfStringPairs(["ab", "ba", "cc"]))               # 1
print(sol.maximumNumberOfStringPairs(["aa", "ab"]))                     # 0
print(sol.maximumNumberOfStringPairs(["ab", "ab", "ba", "aa", "aa", "aa"]))   # mở rộng có trùng: 1 + 1 = 2

2
1
0
2


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 (`set` đã gặp) trên `words = ["cd","ac","dc","ca","zz"]`:

| j | `w` | `w[::-1]` | `seen` trước | bạn đã có? | `count` sau | `seen` sau |
|---|---|---|---|---|---|---|
| 0 | `cd` | `dc` | `{}` | không | 0 | `{cd}` |
| 1 | `ac` | `ca` | `{cd}` | không | 0 | `{cd, ac}` |
| 2 | `dc` | `cd` | `{cd, ac}` | **có** | 1 | `{cd, ac, dc}` |
| 3 | `ca` | `ac` | `{cd, ac, dc}` | **có** | 2 | `{cd, ac, dc, ca}` |
| 4 | `zz` | `zz` | `{cd, ac, dc, ca}` | không | 2 | `{..., zz}` |

Đáp án `2`. Lưu ý ở bước `j = 4`: `"zz"` kiểm tra **trước khi** thêm, nên không thấy chính nó.

In [20]:
def trace_seen(words):
    seen = set()
    count = 0
    for j, w in enumerate(words):
        r = w[::-1]
        has = r in seen
        if has:
            count += 1
        print(f"j={j}, w={w!r}, rev={r!r}, seen_trước={sorted(seen)}, có bạn={has}, count={count}")
        seen.add(w)
    print("kết quả:", count)
    return count

trace_seen(["cd", "ac", "dc", "ca", "zz"])
trace_seen(["ab", "ba", "cc"])

j=0, w='cd', rev='dc', seen_trước=[], có bạn=False, count=0
j=1, w='ac', rev='ca', seen_trước=['cd'], có bạn=False, count=0
j=2, w='dc', rev='cd', seen_trước=['ac', 'cd'], có bạn=True, count=1
j=3, w='ca', rev='ac', seen_trước=['ac', 'cd', 'dc'], có bạn=True, count=2
j=4, w='zz', rev='zz', seen_trước=['ac', 'ca', 'cd', 'dc'], có bạn=False, count=2
kết quả: 2
j=0, w='ab', rev='ba', seen_trước=[], có bạn=False, count=0
j=1, w='ba', rev='ab', seen_trước=['ab'], có bạn=True, count=1
j=2, w='cc', rev='cc', seen_trước=['ab', 'ba'], có bạn=False, count=1
kết quả: 1


1

In [21]:
def trace_brute(words):
    n = len(words)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            ok = words[i] == words[j][::-1]
            print(f"({i},{j}): {words[i]!r} vs đảo({words[j]!r})={words[j][::-1]!r} -> {ok}")
            if ok:
                count += 1
    print("kết quả:", count)
    return count

trace_brute(["ab", "ba", "cc"])

(0,1): 'ab' vs đảo('ba')='ab' -> True
(0,2): 'ab' vs đảo('cc')='cc' -> False
(1,2): 'ba' vs đảo('cc')='cc' -> False
kết quả: 1


1

## 9. Code nộp LeetCode

Dùng Cách 2 (`set`, một lượt, dễ đọc, `O(n)`).

In [22]:
from typing import List

class Solution:
    def maximumNumberOfStringPairs(self, words: List[str]) -> int:
        seen = set()
        count = 0
        for w in words:
            if w[::-1] in seen:      # bạn đảo ngược đã xuất hiện trước đó
                count += 1
            seen.add(w)              # thêm SAU khi kiểm tra
        return count

In [23]:
print(Solution().maximumNumberOfStringPairs(["cd", "ac", "dc", "ca", "zz"]))   # 2
print(Solution().maximumNumberOfStringPairs(["ab", "ba", "cc"]))               # 1
print(Solution().maximumNumberOfStringPairs(["aa", "ab"]))                     # 0

2
1
0


## 10. Vì sao thuật toán đúng

**Bước 1 — đáp án chính là số cặp thoả điều kiện.** Giả sử có hai cặp thoả điều kiện chung một chuỗi, ví dụ `(i, j)` và `(i, k)` với `j != k`. Khi đó `words[j] = rev(words[i]) = words[k]`, mâu thuẫn vì các chuỗi đôi một khác nhau. Tương tự cho các trường hợp chung ở vị trí khác. Vậy các cặp thoả điều kiện **không chung phần tử**, ta có thể chọn **tất cả**, và số cặp tối đa bằng số cặp thoả điều kiện.

**Bước 2 — thuật toán `seen` đếm đúng số cặp.** Xét cặp `(i, j)`, `i < j`, `words[i] = rev(words[j])`. Khi duyệt tới `j`, chuỗi `words[i]` đã được thêm vào `seen` (vì `i < j`), nên điều kiện `rev(words[j]) in seen` đúng ⇒ cộng 1. Ngược lại nếu bước `j` cộng 1 thì tồn tại `i < j` với `words[i] = rev(words[j])`, tức là một cặp thật. Hai điều trên cho song ánh giữa các lần cộng và các cặp.

**Bước 3 — chuỗi đối xứng.** Nếu `w = rev(w)` (như `"zz"`), ta kiểm tra `w in seen` **trước khi** thêm `w`. Vì các chuỗi khác nhau, `w` chưa từng xuất hiện, kiểm tra cho `False`; không đếm sai.

**Bước 4 — đối chiếu.** Cả 5 cách khớp nhau trên mọi input ngẫu nhiên ở mục 12; Cách 1 là bản dịch thẳng của đề nên làm oracle.

## 11. Phân tích Time / Space Complexity

`n = len(words)`; mỗi chuỗi dài 2 nên thao tác trên một chuỗi là `O(1)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force mọi cặp | `O(n²)` | `O(1)` | đơn giản nhất |
| 2. `set` đã gặp | `O(n)` | `O(n)` | một lượt duyệt |
| 3. `w < rev(w)` + set | `O(n)` | `O(n)` | không cần thứ tự duyệt |
| 4. Bảng 26×26 | `O(n)` | `O(1)` | `26 × 26 = 676` ô cố định |
| 5. `Counter` | `O(n)` | `O(n)` | đúng cả khi có trùng |

Với `n <= 50`, mọi cách chạy tức thì. Điều quan trọng là **ý tưởng** thay tìm kiếm bằng tra cứu.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: một chuỗi; chỉ có chuỗi đối xứng; một cặp ở xa nhau; mọi cặp đảo ngược ("ab","ba","cd","dc",...); thứ tự bạn đứng trước hay sau; chuỗi đối xứng lẫn trong cặp.

In [24]:
sols = [SolutionBrute(), SolutionSeen(), SolutionLess(), SolutionGrid(), SolutionCounter(), Solution()]
cases = [
    ("ví dụ 1", ["cd", "ac", "dc", "ca", "zz"], 2),
    ("ví dụ 2", ["ab", "ba", "cc"], 1),
    ("ví dụ 3", ["aa", "ab"], 0),
    ("một chuỗi", ["ab"], 0),
    ("một chuỗi đối xứng", ["aa"], 0),
    ("hai chuỗi bạn nhau", ["ab", "ba"], 1),
    ("bạn đứng trước", ["ba", "ab"], 1),
    ("chỉ có đối xứng", ["aa", "bb", "cc", "dd"], 0),
    ("bạn ở xa nhau", ["ab", "xx", "yy", "zz", "ba"], 1),
    ("hai cặp lồng nhau", ["ab", "cd", "dc", "ba"], 2),
    ("mọi cặp có đủ", ["ab", "ba", "cd", "dc", "ef", "fe"], 3),
    ("không có bạn", ["ab", "cd", "ef"], 0),
]
for name, words, exp in cases:
    for s in sols:
        got = s.maximumNumberOfStringPairs(words[:])
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 2
PASS ví dụ 2 -> 1
PASS ví dụ 3 -> 0
PASS một chuỗi -> 0
PASS một chuỗi đối xứng -> 0
PASS hai chuỗi bạn nhau -> 1
PASS bạn đứng trước -> 1
PASS chỉ có đối xứng -> 0
PASS bạn ở xa nhau -> 1
PASS hai cặp lồng nhau -> 2
PASS mọi cặp có đủ -> 3
PASS không có bạn -> 0


### 12.2. Phản ví dụ: ghép chuỗi đối xứng với chính nó

`words = ["aa"]`, đáp án `0`. Một lời giải sai (thêm trước, kiểm tra sau) sẽ trả `1`. Test của ta bắt được lỗi này (xem mục 13).

### 12.3. Đối chiếu ngẫu nhiên (hợp lệ theo đề)

Sinh `words` gồm các chuỗi **khác nhau** có độ dài 2 lấy từ bảng chữ cái nhỏ (để có nhiều cặp đảo ngược) rồi trộn thứ tự. Độ dài `1 <= n <= 50`. So 5 cách với Cách 1.

In [25]:
import random
from itertools import product
random.seed(2744)

for _ in range(600):
    k = random.choice([2, 3, 4, 5, 8])                       # kích thước bảng chữ cái con
    letters = random.sample("abcdefghijklmnopqrstuvwxyz", k)
    universe = ["".join(p) for p in product(letters, repeat=2)]
    n = random.randint(1, min(50, len(universe)))
    words = random.sample(universe, n)                        # đôi một khác nhau
    exp = SolutionBrute().maximumNumberOfStringPairs(words[:])
    for s in sols:
        got = s.maximumNumberOfStringPairs(words[:])
        assert got == exp, (type(s).__name__, words, got, exp)
    assert 0 <= exp <= n // 2
    snapshot = words[:]
    Solution().maximumNumberOfStringPairs(words)
    assert words == snapshot          # không làm hỏng dữ liệu vào
print("600 ca ngẫu nhiên: 6 cách khớp nhau, 0 <= đáp án <= n//2")

600 ca ngẫu nhiên: 6 cách khớp nhau, 0 <= đáp án <= n//2


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm một cặp hai lần (duyệt mọi `i != j`)

In [26]:
def sai_dem_hai_lan(words: List[str]) -> int:
    count = 0
    for i in range(len(words)):
        for j in range(len(words)):          # sai: j chạy từ 0, không phải i + 1
            if i != j and words[i] == words[j][::-1]:
                count += 1
    return count

w = ["ab", "ba", "cc"]
print("sai :", sai_dem_hai_lan(w))                              # 2
print("đúng:", Solution().maximumNumberOfStringPairs(w))        # 1

sai : 2
đúng: 1


**Vì sao sai:** cặp `(0, 1)` và cặp `(1, 0)` là cùng một cặp. Đề yêu cầu `i < j`, nên `j` phải chạy từ `i + 1`.

### Lỗi 2: Thêm vào `set` trước khi kiểm tra

In [27]:
def sai_them_truoc(words: List[str]) -> int:
    seen = set()
    count = 0
    for w in words:
        seen.add(w)                          # sai: thêm trước
        if w[::-1] in seen:
            count += 1
    return count

print("sai :", sai_them_truoc(["aa", "ab"]))                      # 1
print("đúng:", Solution().maximumNumberOfStringPairs(["aa", "ab"]))  # 0

sai : 1
đúng: 0


**Vì sao sai:** với chuỗi đối xứng như `"aa"`, ngay sau khi thêm `"aa"` vào `seen`, ta thấy `"aa"[::-1] = "aa"` có trong `seen` và đếm nhầm một cặp "với chính nó". Phải **kiểm tra trước, thêm sau**.

### Lỗi 3: Dùng `reversed(w)` rồi so sánh với chuỗi

In [28]:
print(reversed("cd"))                    # <reversed object at ...> — là iterator, KHÔNG phải chuỗi
print(reversed("cd") == "dc")            # False
print("".join(reversed("cd")) == "dc")   # True (cách đúng nếu muốn dùng reversed)
print("cd"[::-1] == "dc")                # True (cách ngắn gọn nhất)

False
True
True


**Vì sao sai:** `reversed(...)` trả về một **iterator**; so sánh nó với chuỗi luôn `False`. Hoặc dùng `s[::-1]`, hoặc `"".join(reversed(s))`.

### Lỗi 4: So sánh `words[i] == words[j]` quên đảo ngược

In [29]:
def sai_quen_dao(words: List[str]) -> int:
    n, count = len(words), 0
    for i in range(n):
        for j in range(i + 1, n):
            if words[i] == words[j]:         # sai: thiếu [::-1]
                count += 1
    return count

w = ["cd", "ac", "dc", "ca", "zz"]
print("sai :", sai_quen_dao(w))                                 # 0
print("đúng:", Solution().maximumNumberOfStringPairs(w))        # 2

sai : 0
đúng: 2


**Vì sao sai:** các chuỗi khác nhau nên `words[i] == words[j]` không bao giờ đúng với `i != j`; ta cần so với **đảo ngược**.

### Lỗi 5: Quên rằng đề đảm bảo các chuỗi khác nhau

In [30]:
# Nếu giả sử có trùng (ngoài đề), code "seen" thường cho đáp án khác "ghép tối đa" thật sự:
data = ["ab", "ba", "ba"]
print("seen   :", Solution().maximumNumberOfStringPairs(data))          # 2 (đếm cả hai "ba" với cùng "ab")
print("Counter:", SolutionCounter().maximumNumberOfStringPairs(data))   # 1 (mỗi chuỗi chỉ thuộc một cặp)

seen   : 2
Counter: 1


**Bài học:** code nộp dựa vào điều kiện của đề (các chuỗi khác nhau). Khi đề bỏ điều kiện này, hãy dùng `Counter` hoặc loại bỏ phần tử đã ghép.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tìm cặp `(i, j)` mà `a[i]` liên hệ với `a[j]` qua một phép biến đổi cố định" → **duyệt một lượt và dùng `set` / `dict`** giống Two Sum: `needed = biến_đổi(x)`; kiểm tra `needed in seen` rồi mới `seen.add(x)`.
- Chuỗi dài 2 ⇒ có thể dùng **bảng 26×26** hoặc ghép thành số `26 * a + b` thay cho chuỗi.
- Nếu đề nói "các phần tử đôi một khác nhau", hãy xem đó là **gợi ý đơn giản hoá** (mỗi phần tử có tối đa một bạn).
- Chuỗi **đối xứng** (palindrome) luôn là trường hợp biên cần nghĩ tới khi đề nói đảo ngược.
- Muốn mỗi cặp đếm đúng một lần: hoặc `i < j`, hoặc "chỉ đếm phần tử nhỏ hơn trong cặp".

## 15. Biến thể của bài

1. **Chuỗi có độ dài bất kỳ** (không chỉ 2): `w[::-1]` vẫn dùng được; `set` vẫn `O(L)` mỗi phép băm.
2. **Cho phép trùng** (bài ghép tối đa): dùng `Counter` như Cách 5.
3. **Hai chuỗi là hoán vị của nhau** (anagram): thay `w[::-1]` bằng `"".join(sorted(w))` làm khoá.
4. **Đếm số cặp `(i, j)`** (không giới hạn mỗi chuỗi một cặp) khi có trùng: đáp án `sum(cnt[w] * cnt[rev(w)])` với xử lý cẩn thận chuỗi đối xứng.
5. **Trả về chính các cặp** (danh sách chỉ số): giữ `dict` từ chuỗi sang chỉ số thay vì `set`.
6. **Palindrome pairs** (LeetCode 336): ghép hai chuỗi bất kỳ thành palindrome, khó hơn nhiều nhưng cùng tinh thần tra cứu băm.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi cặp `i < j` | `O(n²)` | `O(1)` | `n` nhỏ, làm oracle |
| 2. `set` đã gặp | kiểm tra bạn đảo ngược trước, thêm sau | `O(n)` | `O(n)` | **code nộp**, dễ đọc |
| 3. `w < rev(w)` + set | chỉ đếm thành viên nhỏ hơn | `O(n)` | `O(n)` | ngắn gọn, bất biến |
| 4. Bảng 26×26 | ô `(a,b)`, kiểm tra ô `(b,a)` | `O(n)` | `O(1)` | chuỗi dài 2, chữ thường |
| 5. `Counter` | `min(cnt[w], cnt[rev])`, đối xứng `cnt//2` | `O(n)` | `O(n)` | có chuỗi trùng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"xyz"[::-1]`, `"ab" < "ba"`, `sum(w < w[::-1] for w in ["ab","ba","aa"])`, `len({"ab","ba","ab"})`.

**Bài 2 (tính tay).** Với `words = ["ab","ba","cd","dc","ef"]`, lập bảng `seen` qua từng bước theo Cách 2 và nêu đáp án.

**Bài 3 (điền TODO).** Hoàn thành `pairs_todo(words)` theo Cách 2.

In [31]:
def pairs_todo(words):
    seen = set()
    count = 0
    for w in words:
        # TODO 1: nếu đảo ngược của w đã có trong seen thì count += 1
        # TODO 2: thêm w vào seen (sau khi kiểm tra)
        pass
    return count

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `pairs_grid(words)` bằng bảng 26×26 (Cách 4) mà không nhìn code mẫu. Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `pair_indices(words)` trả về **danh sách các cặp chỉ số** `(i, j)` với `i < j`, `words[i] == words[j][::-1]`. (b) Viết `max_pairs_with_duplicates(words)` đúng cả khi mảng có chuỗi trùng (mỗi chuỗi chỉ thuộc một cặp) mà **không** dùng `Counter`, chỉ dùng `dict` thường.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"xyz"[::-1] = "zyx"`; `"ab" < "ba"` là `True`; chỉ `"ab"` thoả `w < rev(w)` nên tổng `1`; set có 2 phần tử. Bài 2: `ab` (không bạn) → thêm; `ba` (bạn `ab` đã có) → +1; `cd` → thêm; `dc` → +1; `ef` → không.
- **Mức 2 (vừa):** Bài 3: `if w[::-1] in seen: count += 1` rồi `seen.add(w)`. Bài 4: `grid[ord(w[1]) - 97][ord(w[0]) - 97]` là ô cần kiểm tra, `grid[ord(w[0]) - 97][ord(w[1]) - 97] = True` là ô cần ghi.
- **Mức 3 (sâu):** Bài 5(a): giữ `dict` từ chuỗi sang **danh sách** chỉ số đã gặp; tại `j`, mỗi `i` trong `d.get(w[::-1], [])` tạo cặp `(i, j)`. Bài 5(b): `d = {}`; mỗi lần gặp `w`, nếu `d.get(rev(w), 0) > 0` thì ghép (giảm `d[rev]`, tăng đếm) ngược lại `d[w] = d.get(w, 0) + 1` — tham lam đúng vì ghép sớm không làm xấu đi.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert "xyz"[::-1] == "zyx"
assert ("ab" < "ba") is True
assert sum(w < w[::-1] for w in ["ab", "ba", "aa"]) == 1
assert len({"ab", "ba", "ab"}) == 2

# Bài 2
assert Solution().maximumNumberOfStringPairs(["ab", "ba", "cd", "dc", "ef"]) == 2

# Bài 3
def pairs_todo_ans(words):
    seen = set()
    count = 0
    for w in words:
        if w[::-1] in seen:
            count += 1
        seen.add(w)
    return count

assert pairs_todo_ans(["cd", "ac", "dc", "ca", "zz"]) == 2
assert pairs_todo_ans(["ab", "ba", "cc"]) == 1
assert pairs_todo_ans(["aa", "ab"]) == 0

# Bài 4
def pairs_grid(words):
    grid = [[False] * 26 for _ in range(26)]
    count = 0
    for w in words:
        a, b = ord(w[0]) - 97, ord(w[1]) - 97
        if grid[b][a]:
            count += 1
        grid[a][b] = True
    return count

assert pairs_grid(["cd", "ac", "dc", "ca", "zz"]) == 2
assert pairs_grid(["ab", "ba", "cc"]) == 1
assert pairs_grid(["aa", "ab"]) == 0

# Bài 5(a)
def pair_indices(words):
    d = {}
    res = []
    for j, w in enumerate(words):
        for i in d.get(w[::-1], []):
            res.append((i, j))
        d.setdefault(w, []).append(j)
    return res

assert pair_indices(["cd", "ac", "dc", "ca", "zz"]) == [(0, 2), (1, 3)]
assert pair_indices(["aa", "ab"]) == []

# Bài 5(b): có trùng, ghép tối đa bằng dict thường
def max_pairs_with_duplicates(words):
    d = {}
    count = 0
    for w in words:
        r = w[::-1]
        if d.get(r, 0) > 0:
            d[r] -= 1
            count += 1
        else:
            d[w] = d.get(w, 0) + 1
    return count

assert max_pairs_with_duplicates(["ab", "ba", "ba"]) == 1
assert max_pairs_with_duplicates(["aa", "aa", "aa"]) == 1
assert max_pairs_with_duplicates(["ab", "ab", "ba", "aa", "aa", "aa"]) == SolutionCounter().maximumNumberOfStringPairs(["ab", "ab", "ba", "aa", "aa", "aa"])
random.seed(1)
for _ in range(300):
    words = ["".join(random.choice("ab") for _ in range(2)) for _ in range(random.randint(1, 12))]
    assert max_pairs_with_duplicates(words) == SolutionCounter().maximumNumberOfStringPairs(words), words
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (duyệt một lượt, kiểm tra bạn trước, thêm sau)

```python
from typing import List

class Solution:
    def maximumNumberOfStringPairs(self, words: List[str]) -> int:
        seen = set()
        count = 0
        for w in words:
            if w[::-1] in seen:   # bạn đảo ngược đã có
                count += 1
            seen.add(w)           # thêm SAU khi kiểm tra
        return count
```

**Checklist**

- [ ] Hiểu cặp đòi hỏi `i < j` và `words[i] == đảo(words[j])`?
- [ ] Dùng `w[::-1]` (không phải `reversed(w)` so với chuỗi)?
- [ ] Kiểm tra `in seen` **trước**, `seen.add` **sau**?
- [ ] Nhớ chuỗi đối xứng như `"zz"` không ghép với chính nó?
- [ ] Biết đề đảm bảo chuỗi khác nhau nên "đếm cặp" = "ghép tối đa"?
- [ ] Đã thử một chuỗi, chỉ có đối xứng, cặp đứng trước/sau?
- [ ] Signature đúng `maximumNumberOfStringPairs(self, words: List[str]) -> int` và có `from typing import List`?